# `ase.optimize`

Together with a calculator, the class `ase.optimize.optimize.Optimizer` offers all the functionality needed for local structure optimization. ASE also provides a global search algorithms (basin hopping and minima hopping) and a transition state search algorithm, but those are not our topic.

The selection of built-in local algorithms include [BFGS](https://ase-lib.org/ase/optimize.html#ase.optimize.LBFGS), [LBFGS](https://ase-lib.org/ase/optimize.html#ase.optimize.LBFGS), and [FIRE](https://ase-lib.org/ase/optimize.html#ase.optimize.FIRE). Choosing the best-fitting algorithm is yet another topic we could spend a lot of time on. Typically, `LBFGS` is a good first choice.

In principle, any `Calculator` can be used with the `Optimizer`. For this introduction, we bring back the Lennard-Jones calculator, later in the workshop we will be using GPAW and MACE instead.

### Example: 13-atom Lennard-Jones cluster

We again use Ar atoms, but set the structure- and calculator parameters to get a minimum close to the literature value of -44.326801 LJ.

In [1]:
import numpy as np
from ase.calculators.lj import LennardJones
from ase.cluster import Icosahedron
from ase.optimize import LBFGS

# build a 13-atom cluster
lj = Icosahedron('Ar', noshells=2, latticeconstant=1.5)
calc = LennardJones(sigma=1.0, epsilon=1.0)
lj.calc = calc
energy = lj.get_potential_energy()

LBFGS(lj, logfile=None).run(fmax=1e-8)

print(
    f"Energy for the relaxed initial cluster is {lj.get_potential_energy():.6f} LJ "
    f"(down from {energy:.6f} LJ)."
)

Energy for the relaxed initial cluster is -43.899405 LJ (down from -43.196306 LJ).


In [2]:
from numpy.random import default_rng
from ase.visualize import view

def add_noise(atoms, scale=0.2, rng_seed=4711, showme=False):
    rng = default_rng(seed=rng_seed)
    
    noisy_atoms = atoms.copy()
    noise = rng.normal(size=lj.positions.shape, scale=scale)
    noisy_atoms.set_positions(atoms.get_positions() + noise)

    if showme:
        view([atoms, noisy_atoms])
    
    return noisy_atoms

In [3]:
scale = 0.25
noisy_lj = add_noise(lj, scale=scale, showme=True)

noisy_lj.calc = calc
print(f"With added noise (scale={scale:.2f}) the energy is {noisy_lj.get_potential_energy():.2f} LJ.")

With added noise (scale=0.25) the energy is 821.85 LJ.


Can we get this back to our original minimum?

In [4]:
import os
import pathlib
import ase.io
from ase.optimize import LBFGS

output_path = pathlib.Path(f"{os.environ["HOME"]}/workshop/ase_intro")
output_path.mkdir(exist_ok=True,  # no error if the directory already exists
                  parents=True,   # create nested directories if needed
                 )

relaxed_lj = noisy_lj.copy()
relaxed_lj.calc = calc
opt = LBFGS(relaxed_lj, trajectory=output_path / 'lj_cluster.traj')
opt.run(fmax=0.01)

print(f"The relaxed setup has an energy of {relaxed_lj.get_potential_energy():.6f} LJ.")

       Step     Time          Energy          fmax
LBFGS:    0 15:00:05      821.854831    14459.022281
LBFGS:    1 15:00:05       27.959399      291.593226
LBFGS:    2 15:00:05      -24.365834       31.058074
LBFGS:    3 15:00:05      -26.311349       22.458018
LBFGS:    4 15:00:05      -23.991209       82.796429
LBFGS:    5 15:00:05      -30.673968       22.867044
LBFGS:    6 15:00:05      -28.933379       40.516709
LBFGS:    7 15:00:05      -31.932197       10.118071
LBFGS:    8 15:00:05      -34.627438       14.541946
LBFGS:    9 15:00:05      -36.048901       16.361238
LBFGS:   10 15:00:05      -37.188598        7.909672
LBFGS:   11 15:00:05      -38.968409       12.752223
LBFGS:   12 15:00:05      -35.275678       52.120557
LBFGS:   13 15:00:05      -38.008723       25.131006
LBFGS:   14 15:00:05      -34.492344       27.996572
LBFGS:   15 15:00:05      -36.765802        9.152837
LBFGS:   16 15:00:05      -39.982081       17.910376
LBFGS:   17 15:00:05      -40.037145       16.33

### Tasks

1. Visualize the trajectory, play with the settings to better see the forces.
2. Try adding more noise and see if you can get the LBFGS optimizer to fail. Why does it fail?
3. For the failing optimization, try a different optimizer.